In [3]:
"""
Brain Age Prediction — Full Pipeline
=====================================
Single script that does everything:
  1. Load .netts timeseries + Smith-10 CSV features
  2. Detrend + Global Signal Regression per subject
  3. Compute Fisher-z FC matrix (164 x 164 → 13,366 edges)
  4. Select top-1,000 FC edges by correlation with age
  5. Concatenate with CSV features → SVR (C=15, ε=0.1, rbf)
  6. Save all .npy artefacts
  7. Generate 8-panel performance dashboard

Requirements:
    pip install scikit-learn xgboost numpy pandas matplotlib scipy

Usage:
    python brain_age_pipeline.py
"""

import re
import zipfile

import matplotlib
matplotlib.use("Agg")
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from numpy.linalg import lstsq
from scipy.stats import norm
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVR

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION  — update paths if needed
# ─────────────────────────────────────────────────────────────────────────────
NETTS_ZIP    = "netts_files.zip"
CSV_PATH     = "cveda_z_smith10(in).csv"
N_REGIONS    = 164
TOP_K_FC     = 1000
SVR_C        = 15
SVR_EPS      = 0.1
N_FOLDS      = 5
RANDOM_STATE = 42
OUT_PLOT     = "mae_dashboard.png"

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — Load CSV metadata
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("STEP 1: Loading CSV metadata ...")
print("=" * 60)

df = pd.read_csv(CSV_PATH)
df["sex_enc"]  = LabelEncoder().fit_transform(df["sex"])
df["site_enc"] = LabelEncoder().fit_transform(df["site"])

print(f"  Subjects in CSV : {len(df)}")
print(f"  Age range       : {df['age'].min():.1f} – {df['age'].max():.1f} yrs")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — Build subject → filename map from zip
# ─────────────────────────────────────────────────────────────────────────────
print("\nSTEP 2: Indexing .netts files ...")

z            = zipfile.ZipFile(NETTS_ZIP)
netts_files  = [n for n in z.namelist() if n.endswith(".netts")]
netts_map    = {
    "sub-" + re.search(r"sub-(\d+)", n).group(1): n
    for n in netts_files
}

df_m = df[df["sub_id"].isin(netts_map)].reset_index(drop=True)
print(f"  .netts files found : {len(netts_files)}")
print(f"  Matched subjects   : {len(df_m)}")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — Compute FC matrices (detrend + GSR + Fisher-z)
# ─────────────────────────────────────────────────────────────────────────────
print("\nSTEP 3: Computing FC matrices (detrend + GSR) ...")

def preprocess_and_fc(ts: np.ndarray) -> np.ndarray:
    """
    ts : (T, R)  raw BOLD signal
    Returns upper-triangle Fisher-z FC vector  shape (R*(R-1)/2,)
    """
    T = ts.shape[0]

    # 1. Linear detrend per region
    t      = np.arange(T, dtype=np.float32)
    t      = (t - t.mean()) / t.std()
    design = np.column_stack([np.ones(T), t])           # (T, 2)
    beta   = lstsq(design, ts, rcond=None)[0]           # (2, R)
    ts     = ts - design @ beta

    # 2. Global signal regression
    gs      = ts.mean(axis=1, keepdims=True)            # (T, 1)
    beta_gs = lstsq(gs, ts, rcond=None)[0]              # (1, R)
    ts      = ts - gs @ beta_gs

    # 3. Standardise
    ts -= ts.mean(axis=0)
    ts /= (ts.std(axis=0) + 1e-9)

    # 4. Pearson correlation → Fisher-z
    corr = (ts.T @ ts) / (T - 1)                       # (R, R)
    np.clip(corr, -0.9999, 0.9999, out=corr)
    np.arctanh(corr, out=corr)

    return corr[np.triu_indices(corr.shape[0], k=1)]

valid_sids = []
FC_list    = []

for i, sid in enumerate(df_m["sub_id"]):
    raw   = z.read(netts_map[sid]).decode("utf-8")
    lines = raw.strip().split("\n")

    if len(lines[0].split()) != N_REGIONS:          # skip incomplete
        continue

    ts = np.array(
        [[float(x) for x in line.split()] for line in lines],
        dtype=np.float32
    )
    FC_list.append(preprocess_and_fc(ts))
    valid_sids.append(sid)

    if (i + 1) % 200 == 0:
        print(f"  processed {i + 1} / {len(df_m)}")

FC   = np.array(FC_list, dtype=np.float32)           # (N, 13366)
df_v = df_m[df_m["sub_id"].isin(valid_sids)].reset_index(drop=True)
y    = df_v["age"].values

print(f"  FC matrix shape : {FC.shape}")
print(f"  FC mean / std   : {FC.mean():.4f} / {FC.std():.4f}")
print(f"  Final subjects  : {len(y)}")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 — Feature engineering
# ─────────────────────────────────────────────────────────────────────────────
print(f"\nSTEP 4: Selecting top-{TOP_K_FC} FC edges ...")

selector = SelectKBest(f_regression, k=TOP_K_FC)
FC_top   = selector.fit_transform(FC, y)              # (N, 1000)

csv_feat_cols = [c for c in df_v.columns if c not in ["sub_id", "age", "sex", "site"]]
X_csv         = df_v[csv_feat_cols].values            # (N, ~48)
X_combined    = np.hstack([X_csv, FC_top])            # (N, ~1048)
X_scaled      = StandardScaler().fit_transform(X_combined)

print(f"  CSV features    : {X_csv.shape[1]}")
print(f"  Combined shape  : {X_combined.shape}")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 5 — Cross-validated SVR
# ─────────────────────────────────────────────────────────────────────────────
print(f"\nSTEP 5: Running {N_FOLDS}-fold SVR (C={SVR_C}, ε={SVR_EPS}) ...")

kf    = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
model = SVR(C=SVR_C, epsilon=SVR_EPS, kernel="rbf", gamma="scale")
preds = cross_val_predict(model, X_scaled, y, cv=kf)

mae_final = mean_absolute_error(y, preds)
print(f"\n  ✅  Final MAE (5-fold CV) = {mae_final:.4f} years")

# Per-fold MAEs
fold_maes = []
for tr, te in kf.split(X_scaled):
    m = SVR(C=SVR_C, epsilon=SVR_EPS, kernel="rbf", gamma="scale")
    m.fit(X_scaled[tr], y[tr])
    fold_maes.append(mean_absolute_error(y[te], m.predict(X_scaled[te])))
fold_maes = np.array(fold_maes)
print(f"  Per-fold MAEs   : {np.round(fold_maes, 4)}")

# CSV-only SVR (for comparison bar chart)
print("\n  Computing CSV-only SVR for comparison ...")
X_csv_sc = StandardScaler().fit_transform(X_csv)
p_csv    = cross_val_predict(
    SVR(C=7, epsilon=0.3, kernel="rbf", gamma="scale"),
    X_csv_sc, y, cv=kf
)
mae_csv = mean_absolute_error(y, p_csv)
print(f"  CSV-only MAE    : {mae_csv:.4f}")

# Progressive feature-count sweep (for MAE vs k plot)
print("\n  Feature-count sweep ...")
prog_maes = []
for k in [100, 200, 300, 500, 700, 1000]:
    FC_k = SelectKBest(f_regression, k=k).fit_transform(FC, y)
    Xk   = StandardScaler().fit_transform(np.hstack([X_csv, FC_k]))
    pk   = cross_val_predict(
        SVR(C=SVR_C, epsilon=SVR_EPS, kernel="rbf", gamma="scale"),
        Xk, y, cv=kf
    )
    prog_maes.append((k, mean_absolute_error(y, pk)))
    print(f"    k={k:4d}  MAE={prog_maes[-1][1]:.4f}")
prog_maes = np.array(prog_maes)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 6 — Save .npy artefacts
# ─────────────────────────────────────────────────────────────────────────────
print("\nSTEP 6: Saving .npy artefacts ...")

np.save("y_plot.npy",       y)
np.save("preds_plot.npy",   preds)
np.save("fold_maes.npy",    fold_maes)
np.save("prog_maes.npy",    prog_maes)
np.save("p_csv.npy",        p_csv)
df_v.to_csv("df_plot.csv",  index=False)

print("  Saved: y_plot.npy, preds_plot.npy, fold_maes.npy,")
print("         prog_maes.npy, p_csv.npy, df_plot.csv")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 7 — Generate 8-panel dashboard
# ─────────────────────────────────────────────────────────────────────────────
print(f"\nSTEP 7: Generating plots → {OUT_PLOT} ...")

errors     = preds - y
abs_errors = np.abs(errors)
mae_baseline = 3.70
mae_logreg   = 3.59

# ── Style ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor" : "#0f1117",
    "axes.facecolor"   : "#1a1d27",
    "axes.edgecolor"   : "#3a3d4a",
    "axes.labelcolor"  : "#e0e0e0",
    "xtick.color"      : "#aaaaaa",
    "ytick.color"      : "#aaaaaa",
    "text.color"       : "#e0e0e0",
    "grid.color"       : "#2a2d3a",
    "grid.alpha"       : 0.6,
    "font.family"      : "DejaVu Sans",
    "font.size"        : 11,
})
BLUE   = "#4f9cf9"
GREEN  = "#2ecc71"
ORANGE = "#f39c12"
RED    = "#e74c3c"
TEAL   = "#1abc9c"

fig = plt.figure(figsize=(20, 16), facecolor="#0f1117")
fig.suptitle(
    "Brain Age Prediction — Model Performance Dashboard",
    fontsize=20, fontweight="bold", color="white", y=0.98
)
gs = gridspec.GridSpec(
    3, 3, figure=fig,
    hspace=0.45, wspace=0.38,
    left=0.07, right=0.97, top=0.93, bottom=0.06
)

# ── Panel 1: Model Comparison ─────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])

model_names = ["Baseline\n(no model)", "Logistic\nRegression",
               "CSV-only\nSVR", "Final Model\n(SVR + FC)"]
maes        = [mae_baseline, mae_logreg, mae_csv, mae_final]
bar_colors  = [RED, ORANGE, BLUE, GREEN]

bars = ax1.bar(model_names, maes, color=bar_colors, width=0.55,
               edgecolor="#ffffff22", linewidth=0.8, zorder=3)
for bar, mae in zip(bars, maes):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.03,
        f"{mae:.2f}",
        ha="center", va="bottom",
        fontweight="bold", color="white", fontsize=11
    )
ax1.set_ylim(0, max(maes) * 1.25)
ax1.set_ylabel("MAE (years)", fontsize=11)
ax1.set_title("Model Comparison", fontsize=13, fontweight="bold", pad=10)
ax1.grid(axis="y", zorder=0)
ax1.tick_params(axis="x", labelsize=9)

# ── Panel 2: MAE vs Top-K FC Features ────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])

ks    = prog_maes[:, 0].astype(int)
mvals = prog_maes[:, 1]

ax2.plot(ks, mvals, color=BLUE, linewidth=2.5,
         marker="o", markersize=8,
         markerfacecolor=GREEN, markeredgecolor="white",
         markeredgewidth=1.5, zorder=3)
ax2.fill_between(ks, mvals, mvals.max() + 0.05, alpha=0.12, color=BLUE)
ax2.axvline(TOP_K_FC, color=GREEN, linewidth=1.5, linestyle=":",
            alpha=0.8, label=f"Best k = {TOP_K_FC}")

for k, m in zip(ks, mvals):
    ax2.annotate(
        f"{m:.3f}", (k, m),
        textcoords="offset points", xytext=(0, 10),
        ha="center", fontsize=8.5, color="#cccccc"
    )
ax2.set_xlabel("Top-K FC Edges Selected", fontsize=11)
ax2.set_ylabel("MAE (years)", fontsize=11)
ax2.set_title("MAE vs Feature Count\n(FC edges + CSV)",
              fontsize=13, fontweight="bold", pad=10)
ax2.legend(fontsize=9, framealpha=0.3)
ax2.grid(zorder=0)
ax2.set_xticks(ks)
ax2.tick_params(axis="x", labelsize=8, rotation=25)

# ── Panel 3: Per-Fold MAE ─────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])

fold_labels = [f"Fold {i+1}" for i in range(len(fold_maes))]
ax3.bar(fold_labels, fold_maes, color=BLUE, width=0.5,
        edgecolor="#ffffff22", linewidth=0.8, zorder=3)
ax3.axhline(mae_final, color=TEAL, linewidth=2, linestyle="-",
            label=f"Mean MAE = {mae_final:.3f}", zorder=4)

for i, m in enumerate(fold_maes):
    ax3.text(
        i, m + 0.02, f"{m:.3f}",
        ha="center", va="bottom",
        fontweight="bold", color="white", fontsize=10
    )

ax3.set_ylim(0, max(fold_maes) * 1.25)
ax3.set_ylabel("MAE (years)", fontsize=11)
ax3.set_title("Per-Fold MAE\n(5-Fold Cross-Validation)",
              fontsize=13, fontweight="bold", pad=10)
ax3.legend(fontsize=9, framealpha=0.3)
ax3.grid(axis="y", zorder=0)

# ── Panel 4: Actual vs Predicted ─────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, :2])

sc = ax4.scatter(y, preds, c=abs_errors, cmap="RdYlGn_r",
                 s=18, alpha=0.65, linewidths=0, zorder=3)
age_line = np.linspace(y.min() - 1, y.max() + 1, 100)
ax4.plot(age_line, age_line, color="white", linewidth=2,
         linestyle="--", label="Perfect prediction", alpha=0.7, zorder=4)

cb = fig.colorbar(sc, ax=ax4, pad=0.02)
cb.set_label("Absolute Error (years)", color="#cccccc", fontsize=10)
cb.ax.yaxis.set_tick_params(color="#aaaaaa")
plt.setp(cb.ax.yaxis.get_ticklabels(), color="#aaaaaa")

ax4.set_xlabel("Actual Age (years)", fontsize=12)
ax4.set_ylabel("Predicted Age (years)", fontsize=12)
ax4.set_title(
    f"Actual vs Predicted Age  |  MAE = {mae_final:.4f} yrs  |  n = {len(y)}",
    fontsize=13, fontweight="bold", pad=10
)
ax4.legend(fontsize=10, framealpha=0.3)
ax4.grid(zorder=0)

# ── Panel 5: Error Distribution ──────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])

ax5.hist(errors, bins=40, color=BLUE, edgecolor="#ffffff22",
         alpha=0.85, zorder=3, density=True)
mu, sigma = errors.mean(), errors.std()
xx = np.linspace(errors.min(), errors.max(), 200)
ax5.plot(xx, norm.pdf(xx, mu, sigma), color=GREEN, linewidth=2.5,
         label=f"Normal fit\nμ={mu:.2f}, σ={sigma:.2f}")
ax5.axvline(0, color="white", linewidth=1.5, linestyle="--", alpha=0.6)
ax5.axvline( mae_final, color=ORANGE, linewidth=1.5,
             linestyle=":", label=f"±MAE = {mae_final:.2f} yrs")
ax5.axvline(-mae_final, color=ORANGE, linewidth=1.5, linestyle=":")

ax5.set_xlabel("Prediction Error (years)", fontsize=11)
ax5.set_ylabel("Density", fontsize=11)
ax5.set_title("Error Distribution", fontsize=13, fontweight="bold", pad=10)
ax5.legend(fontsize=9, framealpha=0.3)
ax5.grid(zorder=0)

# ── Panel 6: MAE by Age Group ────────────────────────────────────────────────
ax6 = fig.add_subplot(gs[2, 0])

age_bins   = [5, 10, 13, 16, 19, 25]
bin_labels = ["5–10", "10–13", "13–16", "16–19", "19–25"]
bin_maes, bin_counts = [], []
for lo, hi in zip(age_bins[:-1], age_bins[1:]):
    mask = (y >= lo) & (y < hi)
    bin_maes.append(
        mean_absolute_error(y[mask], preds[mask]) if mask.sum() > 0 else 0.0
    )
    bin_counts.append(int(mask.sum()))

ax6.bar(bin_labels, bin_maes, color=BLUE, width=0.55,
        edgecolor="#ffffff22", zorder=3)
for i, (m, cnt) in enumerate(zip(bin_maes, bin_counts)):
    ax6.text(
        i, m + 0.03, f"{m:.2f}\n(n={cnt})",
        ha="center", va="bottom", fontsize=8.5, color="white"
    )

ax6.set_xlabel("Age Group (years)", fontsize=11)
ax6.set_ylabel("MAE (years)", fontsize=11)
ax6.set_title("MAE by Age Group", fontsize=13, fontweight="bold", pad=10)
ax6.grid(axis="y", zorder=0)
ax6.set_ylim(0, max(bin_maes) * 1.35)

# ── Panel 7: MAE by Site ──────────────────────────────────────────────────────
ax7 = fig.add_subplot(gs[2, 1])

sites      = df_v["site"].values
site_names = sorted(df_v["site"].unique())
site_maes, site_counts = [], []
for s in site_names:
    mask = sites == s
    site_maes.append(mean_absolute_error(y[mask], preds[mask]))
    site_counts.append(int(mask.sum()))

ax7.bar(site_names, site_maes, color=BLUE, width=0.55,
        edgecolor="#ffffff22", zorder=3)
for i, (m, cnt) in enumerate(zip(site_maes, site_counts)):
    ax7.text(
        i, m + 0.03, f"{m:.2f}\n(n={cnt})",
        ha="center", va="bottom", fontsize=8.5, color="white"
    )

ax7.set_xlabel("Site", fontsize=11)
ax7.set_ylabel("MAE (years)", fontsize=11)
ax7.set_title("MAE by Acquisition Site",
              fontsize=13, fontweight="bold", pad=10)
ax7.grid(axis="y", zorder=0)
ax7.set_ylim(0, max(site_maes) * 1.35)

# ── Panel 8: Residuals vs Predicted ──────────────────────────────────────────
ax8 = fig.add_subplot(gs[2, 2])

ax8.scatter(preds, errors, c=abs_errors, cmap="RdYlGn_r",
            s=16, alpha=0.6, linewidths=0, zorder=3)
ax8.axhline(0, color="white", linewidth=2, linestyle="--",
            alpha=0.7, zorder=4)
ax8.axhline( mae_final, color=ORANGE, linewidth=1.5, linestyle=":",
             alpha=0.8, label=f"±MAE = {mae_final:.2f} yrs")
ax8.axhline(-mae_final, color=ORANGE, linewidth=1.5, linestyle=":", alpha=0.8)

ax8.set_xlabel("Predicted Age (years)", fontsize=11)
ax8.set_ylabel("Residual (Predicted − Actual)", fontsize=11)
ax8.set_title("Residuals vs Predicted Age",
              fontsize=13, fontweight="bold", pad=10)
ax8.legend(fontsize=9, framealpha=0.3)
ax8.grid(zorder=0)

# ── Save ─────────────────────────────────────────────────────────────────────
plt.savefig(OUT_PLOT, dpi=160, bbox_inches="tight", facecolor="#0f1117")
print(f"\n  Dashboard saved → {OUT_PLOT}")

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"  Subjects used   : {len(y)}")
print(f"  Age range       : {y.min():.1f} – {y.max():.1f} yrs")
print(f"  Baseline MAE    : {mae_baseline:.4f}")
print(f"  Logistic Reg MAE: {mae_logreg:.4f}")
print(f"  CSV-only SVR MAE: {mae_csv:.4f}")
print(f"  Final SVR MAE   : {mae_final:.4f}")
print(f"  Per-fold MAEs   : {np.round(fold_maes, 4)}")
print(f"\n  Files saved:")
print(f"    y_plot.npy, preds_plot.npy, fold_maes.npy")
print(f"    prog_maes.npy, p_csv.npy, df_plot.csv")
print(f"    {OUT_PLOT}")
print("=" * 60)

STEP 1: Loading CSV metadata ...
  Subjects in CSV : 1029
  Age range       : 5.5 – 24.3 yrs

STEP 2: Indexing .netts files ...
  .netts files found : 1024
  Matched subjects   : 1024

STEP 3: Computing FC matrices (detrend + GSR) ...
  processed 200 / 1024
  processed 400 / 1024
  processed 600 / 1024
  processed 800 / 1024
  processed 1000 / 1024
  FC matrix shape : (1016, 13366)
  FC mean / std   : -0.0040 / 0.2892
  Final subjects  : 1016

STEP 4: Selecting top-1000 FC edges ...
  CSV features    : 48
  Combined shape  : (1016, 1048)

STEP 5: Running 5-fold SVR (C=15, ε=0.1) ...

  ✅  Final MAE (5-fold CV) = 2.7771 years
  Per-fold MAEs   : [3.0655 2.7704 2.9792 2.5163 2.5527]

  Computing CSV-only SVR for comparison ...
  CSV-only MAE    : 3.0813

  Feature-count sweep ...
    k= 100  MAE=3.1727
    k= 200  MAE=3.1263
    k= 300  MAE=3.0896
    k= 500  MAE=2.9482
    k= 700  MAE=2.8712
    k=1000  MAE=2.7771

STEP 6: Saving .npy artefacts ...
  Saved: y_plot.npy, preds_plot.npy, f

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR   = "/content/drive/MyDrive/NEE_project"       # folder containing your files
CSV_PATH   = f"{BASE_DIR}/cveda_z_smith10_in_.csv"
ZIP_PATH   = f"{BASE_DIR}/netts_files.zip"
NETTS_DIR  = "/content/Glasser_HCP"                     # local unzip destination
OUT_DIR    = f"{BASE_DIR}/outputs"
# ──────────────────────────────────────────────────────────

os.makedirs(OUT_DIR, exist_ok=True)
print("Drive mounted. Paths set.")
print(f"CSV    : {CSV_PATH}")
print(f"ZIP    : {ZIP_PATH}")
print(f"Output : {OUT_DIR}")

!pip install -q statsmodels --upgrade

import zipfile, shutil

if not os.path.exists(NETTS_DIR):
    print("Unzipping netts files (this takes ~1 min)...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall("/content/")
    extracted = [d for d in os.listdir("/content/") if "Glasser" in d or "netts" in d.lower()]
    print(f"Extracted folders: {extracted}")
else:
    print("Netts directory already exists, skipping unzip.")

n_netts = len([f for f in os.listdir(NETTS_DIR) if f.endswith('.netts')])
print(f"Found {n_netts} .netts files in {NETTS_DIR}")
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from scipy import stats
from scipy.stats import mannwhitneyu
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

N_REGIONS = 360
GROUP_DEFS = [
    ("Group I\n(5–10 yrs)",    5,   10),
    ("Group II\n(10–14 yrs)", 10,   14),
    ("Group III\n(14–18 yrs)",14,   18),
    ("Group IV\n(18–25 yrs)", 18,  26),
]
GROUP_COLORS       = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']
GROUP_LABELS_SHORT = ['5–10', '10–14', '14–18', '18–25']

plt.rcParams.update({
    'font.size': 11, 'axes.titlesize': 13,
    'axes.labelsize': 12, 'font.family': 'DejaVu Sans',
    'figure.dpi': 150,
})
print("Imports done.")

def load_netts(path, n_regions=360): #TXN
    data = np.loadtxt(path)
    if data.shape[0] == n_regions:
        return data.T          # (T × N)
    elif data.shape[1] == n_regions:
        return data            # already (T × N)
    else:
        raise ValueError(f"Unexpected shape {data.shape} in {path}")

def compute_fc_matrix(ts):
    """
    Builds binary adjacency matrix from time series.
    Follows paper exactly:
      1. Pearson r between all region pairs
      2. Fisher r-to-z transform
      3. Set negative z to 0 (keep positive only)
      4. Bonferroni correction: p > 0.05 → 0, else → 1
    Returns: binary (N × N) adjacency matrix
    """
    T, N = ts.shape

    # Step 1: Pearson correlation matrix
    corr = np.corrcoef(ts.T)                      # (N × N)
    np.fill_diagonal(corr, 0)

    # Step 2: Fisher r-to-z transform
    corr_clipped = np.clip(corr, -0.9999, 0.9999)
    z = np.arctanh(corr_clipped)

    # Step 3: Keep positive only (paper: "avoid ambiguous biological explanations")
    z[z < 0] = 0
    r = np.tanh(z)

    # Step 4: Bonferroni correction
    n_pairs      = N * (N - 1) / 2
    alpha_bonf   = 0.05 / n_pairs
    df           = T - 2
    with np.errstate(divide='ignore', invalid='ignore'):
        t_stat = r * np.sqrt(df) / np.sqrt(np.maximum(1 - r**2, 1e-10))
    p_vals = 2 * stats.t.sf(np.abs(t_stat), df=df)

    adj = np.zeros((N, N))
    adj[p_vals <= alpha_bonf] = 1
    np.fill_diagonal(adj, 0)

    return adj


def compute_ec(adj, max_iter=1000, tol=1e-8):
    """
    Power iteration to find the leading eigenvector (Perron vector)
    of the adjacency matrix — this is the Eigenvector Centrality (EC).
    Paper: "using the Perron-Frobenius theorem, the centrality vector
            has strictly positive components."
    Returns: EC vector of length N
    """
    N = adj.shape[0]
    ec = np.ones(N) / N          # uniform initialization
    for _ in range(max_iter):
        ec_new = adj @ ec
        norm = np.linalg.norm(ec_new)
        if norm < 1e-12:
            ec_new = np.ones(N) / N
            break

        ec_new /= norm
        if np.max(np.abs(ec_new - ec)) < tol:
            break
        ec = ec_new
    return np.abs(ec_new)        # ensure positive (Perron-Frobenius)

def compute_nee(ec):
    """
    Energy Probability and Network Eigen-Entropy.

    Paper equations:
      EP_i  = EC_i / Σ EC_j          (Eq. 2)
      NEE   = −Σ EP_i · ln(EP_i)     (Eq. 3)

    Returns: (nee scalar, ep array of length N)
    """
    ep       = ec / ec.sum()
    ep_safe  = np.where(ep > 0, ep, 1e-15)
    nee      = -np.sum(ep_safe * np.log(ep_safe))
    return nee, ep

print("Core functions defined.")
print("  load_netts()    → (T×N) time-series array")
print("  compute_fc()    → binary (N×N) adjacency matrix (Bonferroni)")
print("  compute_ec()    → eigenvector centrality (power iteration)")
print("  compute_nee()   → (NEE scalar, EP array)")

import time

df_meta  = pd.read_csv(CSV_PATH)
sub_ids  = df_meta['sub_id'].tolist()
results  = []
errors   = 0
t0       = time.time()

print(f"Processing {len(sub_ids)} subjects...")
print("This typically takes 15–20 minutes in Colab (1024 subjects × 360 regions).\n")

for i, sub_id in enumerate(sub_ids):
    netts_path = os.path.join(NETTS_DIR, f"{sub_id}_000.netts")
    if not os.path.exists(netts_path):
        errors += 1
        continue

    try:
        ts       = load_netts(netts_path)
        adj      = compute_fc_matrix(ts)
        ec       = compute_ec(adj)
        nee, ep  = compute_nee(ec)

        row = df_meta[df_meta['sub_id'] == sub_id].iloc[0]
        results.append({
            'sub_id' : sub_id,
            'age'    : float(row['age']),
            'sex'    : 1 if row['sex'] == 'M' else 0,
            'site'   : row['site'],
            'meanFD' : float(row.get('mean_FD', 0)),
            'NEE'    : nee,
            'n_edges': int(adj.sum() / 2),
            'ec'     : ec.tolist(),
            'ep'     : ep.tolist(),
        })
    except Exception as e:
        errors += 1

    if (i + 1) % 100 == 0:
        elapsed = time.time() - t0
        eta     = elapsed / (i + 1) * (len(sub_ids) - i - 1)
        print(f"  [{i+1:4d}/{len(sub_ids)}] done={len(results)}, errors={errors}, "
              f"elapsed={elapsed/60:.1f}min, ETA={eta/60:.1f}min")

print(f"\nFinished. Processed: {len(results)}, Errors: {errors}")

# ─── Build DataFrames ──────────────────────────────────────
df        = pd.DataFrame([{k: v for k, v in r.items()
                           if k not in ('ec','ep')} for r in results])
ec_matrix = np.array([r['ec'] for r in results])   # (N_subs × 360)
ep_matrix = np.array([r['ep'] for r in results])   # (N_subs × 360)

# Assign age groups
def assign_group(age):
    for i, (label, lo, hi) in enumerate(GROUP_DEFS):
        if lo <= age < hi:
            return i
    return -1

df['group']  = df['age'].apply(assign_group)
df_valid     = df[df['group'] >= 0].copy()
group_dfs    = [df_valid[df_valid['group'] == i] for i in range(4)]

# Save CSV checkpoint
df_valid.drop(columns=['ec','ep'], errors='ignore').to_csv(
    f"{OUT_DIR}/nee_results.csv", index=False)

print("\nGroup summary:")
for i, (label, lo, hi) in enumerate(GROUP_DEFS):
    g = group_dfs[i]
    print(f"  {label.replace(chr(10),' ')}: n={len(g)}, "
          f"age {g['age'].mean():.1f}±{g['age'].std():.1f}, "
          f"NEE {g['NEE'].mean():.4f}±{g['NEE'].std():.4f}, "
          f"M/F={(g['sex']==1).sum()}/{(g['sex']==0).sum()}")


# ════════════════════════════════════════════════════════════
# CELL 6 — Statistical Analysis
# ════════════════════════════════════════════════════════════

# ─── 1. Multiple Linear Regression ────────────────────────
# Paper Eq. 6:  E = β0 + β1·age + β2·age² + β3·sex + β4·meanFD
# We also add site dummies (multi-site dataset)
df_reg       = df_valid.dropna(subset=['NEE','age','sex','meanFD']).copy()
df_reg['age2'] = df_reg['age'] ** 2
site_dummies = pd.get_dummies(df_reg['site'], prefix='site', drop_first=True)
X = add_constant(df_reg[['age','age2','sex','meanFD']].join(site_dummies)).astype(float)
y = df_reg['NEE'].astype(float)

model     = OLS(y, X).fit()
beta_age  = model.params['age']
beta_age2 = model.params['age2']
p_age     = model.pvalues['age']
p_age2    = model.pvalues['age2']
r2        = model.rsquared

print("─── Regression Results ───────────────────────────────")
print(model.summary().tables[1])
print(f"\nR² = {r2:.4f}")
print(f"age:  β={beta_age:.4f},  p={p_age:.4e}")
print(f"age²: β={beta_age2:.4f}, p={p_age2:.4e}")

# ─── 2. Polynomial fit for scatter plot ───────────────────
ages_all = df_valid['age'].values
nee_all  = df_valid['NEE'].values
coeffs   = np.polyfit(ages_all, nee_all, 2)
age_range = np.linspace(ages_all.min(), ages_all.max(), 200)
nee_fit   = np.polyval(coeffs, age_range)
r2_poly   = np.corrcoef(nee_all, np.polyval(coeffs, ages_all))[0,1]**2

# ─── 3. Mann-Whitney pairwise group comparisons ────────────
print("\n─── Mann-Whitney Group Comparisons ──────────────────")
pair_keys  = [(0,1),(1,2),(2,3)]
mw_results = {}
for (i,j) in pair_keys:
    stat, p = mannwhitneyu(group_dfs[i]['NEE'].values,
                           group_dfs[j]['NEE'].values,
                           alternative='two-sided')
    mw_results[(i,j)] = p
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
    print(f"  G{i+1} vs G{j+1}: U={stat:.0f}, p={p:.4e} {sig}")

# ─── 4. Regional EP analysis (Mann-Whitney + FDR) ─────────
print("\n─── Regional EP Changes (FDR q<0.05) ───────────────")

def region_mw_test(ep_mat, idx_a, idx_b):
    p_vals, z_vals = [], []
    for r in range(N_REGIONS):
        ep_a = ep_mat[idx_a, r]
        ep_b = ep_mat[idx_b, r]
        stat, p  = mannwhitneyu(ep_a, ep_b, alternative='two-sided')
        n1, n2   = len(ep_a), len(ep_b)
        mu_u     = n1 * n2 / 2
        sigma_u  = np.sqrt(n1 * n2 * (n1 + n2 + 1) / 12)
        z        = (stat - mu_u) / sigma_u
        p_vals.append(p)
        z_vals.append(z)
    reject, p_fdr, _, _ = multipletests(p_vals, method='fdr_bh')
    return np.array(p_vals), np.array(p_fdr), np.array(z_vals), reject

idx_g = [np.where(df_valid['group'].values == i)[0] for i in range(4)]
p12, pfdr12, z12, sig12 = region_mw_test(ep_matrix, idx_g[0], idx_g[1])
p23, pfdr23, z23, sig23 = region_mw_test(ep_matrix, idx_g[1], idx_g[2])
p34, pfdr34, z34, sig34 = region_mw_test(ep_matrix, idx_g[2], idx_g[3])

for label, sig in [("G1 vs G2", sig12), ("G2 vs G3", sig23), ("G3 vs G4", sig34)]:
    print(f"  {label}: {sig.sum()} sig. regions ({100*sig.mean():.2f}%)")

# Group-level mean EP arrays (for histograms)
group_ep_means = [ep_matrix[df_valid['group'].values == i].mean(axis=0)
                  for i in range(4)]

print("\nStatistical analysis complete.")


# ════════════════════════════════════════════════════════════
# CELL 7 — Figure 1: NEE Scatter + Box Plot  (≈ paper Fig. 3)
# ════════════════════════════════════════════════════════════

def draw_sig_bracket(ax, x1, x2, y, p, yrange):
    h = yrange * 0.02
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.2, color='black')
    lbl = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
    ax.text((x1+x2)/2, y+h*1.2, lbl, ha='center', va='bottom', fontsize=11)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.patch.set_facecolor('#FAFAFA')

# ─── Left: scatter + polynomial fit ───────────────────────
ax = axes[0]
for i, (label, lo, hi) in enumerate(GROUP_DEFS):
    mask = df_valid['group'] == i
    ax.scatter(df_valid.loc[mask,'age'], df_valid.loc[mask,'NEE'],
               color=GROUP_COLORS[i], alpha=0.5, s=22,
               label=label.replace('\n',' '), zorder=3)
ax.plot(age_range, nee_fit, 'k-', lw=2.5,
        label=f'Poly fit (R²={r2_poly:.3f})', zorder=5)
ax.plot(age_range, np.polyval(np.polyfit(ages_all, nee_all, 1), age_range),
        'k--', lw=1.5, alpha=0.5, label='Linear fit')
ax.set_xlabel("Age (years)")
ax.set_ylabel("Network Eigen-Entropy (NEE)")
ax.set_title("NEE vs Age — Polynomial Trajectory", fontweight='bold')
ax.legend(fontsize=9, framealpha=0.85)
ax.text(0.04, 0.06,
        f"β_age={beta_age:.4f} (p={p_age:.3e})\n"
        f"β_age²={beta_age2:.4f} (p={p_age2:.3e})\n"
        f"MLR R²={r2:.3f}",
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.9))
ax.grid(axis='y', alpha=0.3)

# ─── Right: box plot with significance brackets ────────────
ax2     = axes[1]
gdata   = [group_dfs[i]['NEE'].values for i in range(4)]
bp      = ax2.boxplot(gdata, patch_artist=True, widths=0.55,
                      medianprops=dict(color='black', lw=2.5),
                      whiskerprops=dict(lw=1.5), capprops=dict(lw=1.5))
for patch, c in zip(bp['boxes'], GROUP_COLORS):
    patch.set_facecolor(c); patch.set_alpha(0.75)

ymax   = max(g.max() for g in gdata)
ymin   = min(g.min() for g in gdata)
yrange_box = ymax - ymin
for k, ((i,j), (x1,x2)) in enumerate(zip(pair_keys, [(1,2),(2,3),(3,4)])):
    draw_sig_bracket(ax2, x1, x2,
                     ymax + yrange_box*0.08*(k+1),
                     mw_results[(i,j)], yrange_box)

ax2.set_xticks([1,2,3,4])
ax2.set_xticklabels(GROUP_LABELS_SHORT, fontsize=10)
ax2.set_xlabel("Age Group (years)")
ax2.set_ylabel("Network Eigen-Entropy (NEE)")
ax2.set_title("NEE Group Comparisons (Mann-Whitney)", fontweight='bold')
ax2.set_ylim(ymin - yrange_box*0.05, ymax + yrange_box*0.45)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout(pad=2)
plt.savefig(f"{OUT_DIR}/Fig1_NEE_scatter_and_boxplot.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved Fig1")


# ════════════════════════════════════════════════════════════
# CELL 8 — Figure 2: EP Histograms & Sorted EP Bar  (≈ paper Fig. 2)
# ════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.patch.set_facecolor('#FAFAFA')
fig.suptitle(
    "Energy Probability (EP) Distribution by Age Group\n"
    "C-VEDA · Glasser 360 parcels  |  Replicates Fan et al. (2017) Fig. 2",
    fontsize=13, fontweight='bold')

for i in range(4):
    label = GROUP_DEFS[i][0].replace('\n', ' ')
    color = GROUP_COLORS[i]
    n_sub = len(group_dfs[i])

    # ─ Top row: sorted EP bar (simulates topographic EP map) ─
    ax_top = axes[0, i]
    sorted_ep = np.sort(group_ep_means[i])
    ax_top.bar(range(N_REGIONS), sorted_ep, color=color, alpha=0.7, width=1.0)
    ax_top.set_title(f"{label}\n(n={n_sub})", fontweight='bold', fontsize=11)
    ax_top.set_xlabel("Regions (sorted by EP value)")
    ax_top.set_ylabel("Mean EP" if i == 0 else "")
    ax_top.set_xlim(0, N_REGIONS)
    ax_top.grid(axis='y', alpha=0.3)

    ep_range = sorted_ep.max() - sorted_ep.min()
    ax_top.text(0.97, 0.97,
                f"Range: {ep_range:.5f}",
                ha='right', va='top', transform=ax_top.transAxes,
                fontsize=8.5,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

    # ─ Bottom row: EP histogram (concentration → dispersion) ─
    ax_bot = axes[1, i]
    ep_flat = ep_matrix[df_valid['group'].values == i].flatten()
    ax_bot.hist(ep_flat, bins=70, color=color, alpha=0.75,
                density=True, edgecolor='white', linewidth=0.3)
    ax_bot.set_xlabel("Energy Probability")
    ax_bot.set_ylabel("Density" if i == 0 else "")
    ax_bot.set_title(f"EP Histogram — {label}", fontsize=10)
    ax_bot.grid(axis='y', alpha=0.3)

    std_ep = ep_flat.std()
    ax_bot.text(0.97, 0.95,
                f"σ = {std_ep:.5f}",
                ha='right', va='top', transform=ax_bot.transAxes,
                fontsize=9,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

plt.tight_layout(pad=1.5)
plt.savefig(f"{OUT_DIR}/Fig2_EP_histograms_by_group.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved Fig2")


# ════════════════════════════════════════════════════════════
# CELL 9 — Figure 3: NEE Trajectory + Violin  (≈ paper Figs 3–4)
# ════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.patch.set_facecolor('#FAFAFA')

# ─── Left: scatter coloured by group + both fits ──────────
ax = axes[0]
for i in range(4):
    mask = df_valid['group'] == i
    ax.scatter(df_valid.loc[mask,'age'], df_valid.loc[mask,'NEE'],
               c=GROUP_COLORS[i], alpha=0.4, s=18,
               label=GROUP_LABELS_SHORT[i]+' yrs', zorder=3)
ax.plot(age_range, nee_fit, 'k-', lw=2.5, zorder=6,
        label=f'Polynomial (R²={r2_poly:.3f})')
ax.plot(age_range, np.polyval(np.polyfit(ages_all, nee_all, 1), age_range),
        'k--', lw=1.5, alpha=0.55, label='Linear')
ax.set_xlabel("Age (years)")
ax.set_ylabel("NEE")
ax.set_title("Whole-Brain NEE Across Development\n(C-VEDA, n=1024)", fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.25)
ax.text(0.04, 0.08,
        f"β_age = {beta_age:.4f}  (p={p_age:.2e})\n"
        f"β_age² = {beta_age2:.4f} (p={p_age2:.2e})\n"
        f"MLR R² = {r2:.3f}",
        transform=ax.transAxes, fontsize=8.5,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

# ─── Right: violin plot ────────────────────────────────────
ax2 = axes[1]
parts = ax2.violinplot([group_dfs[i]['NEE'].values for i in range(4)],
                       positions=[1,2,3,4], widths=0.7,
                       showmeans=True, showmedians=True)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(GROUP_COLORS[i])
    pc.set_alpha(0.6)
parts['cmeans'].set_color('black')
parts['cmedians'].set_color('red')

# Overlay jittered points
rng = np.random.default_rng(42)
for i in range(4):
    yv = group_dfs[i]['NEE'].values
    xv = rng.normal(i+1, 0.055, size=len(yv))
    ax2.scatter(xv, yv, s=5, alpha=0.25, color=GROUP_COLORS[i])

y_top = max(g['NEE'].max() for g in group_dfs)
for k, (i,j) in enumerate(pair_keys):
    p  = mw_results[(i,j)]
    lbl = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
    ax2.text((i+j)/2 + 1.5, y_top - k*0.012,
             f"p={p:.4f} {lbl}", ha='center', fontsize=8.5, color='darkred')

ax2.set_xticks([1,2,3,4])
ax2.set_xticklabels(GROUP_LABELS_SHORT, fontsize=10)
ax2.set_xlabel("Age Group (years)")
ax2.set_ylabel("NEE")
ax2.set_title("NEE Distribution by Developmental Group", fontweight='bold')
ax2.grid(axis='y', alpha=0.25)

plt.tight_layout(pad=2)
plt.savefig(f"{OUT_DIR}/Fig3_NEE_trajectory_and_violin.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved Fig3")

# ════════════════════════════════════════════════════════════
# CELL 10 — Figure 4: Brain Region EP Changes  (≈ paper Fig. 5)
# ════════════════════════════════════════════════════════════

comparison_data = [
    (z12, sig12, "Group I vs II\n(5–10 vs 10–14 yrs)"),
    (z23, sig23, "Group II vs III\n(10–14 vs 14–18 yrs)"),
    (z34, sig34, "Group III vs IV\n(14–18 vs 18–25 yrs)"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
fig.patch.set_facecolor('#FAFAFA')
fig.suptitle(
    "Brain Regions with Significant EP Changes (FDR-corrected, q<0.05)\n"
    "Z-value: positive (red) = EP increase, negative (blue) = EP decrease",
    fontsize=13, fontweight='bold')

for ax, (z_vals, sig_mask, title) in zip(axes, comparison_data):
    if sig_mask.sum() == 0:
        ax.text(0.5, 0.5, "No significant regions\nafter FDR correction",
                ha='center', va='center', transform=ax.transAxes,
                fontsize=12, color='grey')
        ax.set_title(title.replace('\n', '\n'), fontweight='bold', fontsize=10)
        continue

    sig_idx = np.where(sig_mask)[0]
    z_sig   = z_vals[sig_mask]
    colors  = ['#D32F2F' if z > 0 else '#1565C0' for z in z_sig]
    order   = np.argsort(np.abs(z_sig))[::-1]
    top_n   = min(40, len(order))
    plot_ord = order[:top_n]

    ax.barh(range(top_n), z_sig[plot_ord[::-1]],
            color=[colors[j] for j in plot_ord[::-1]],
            alpha=0.85, edgecolor='white', linewidth=0.5)
    ax.axvline(0, color='black', lw=0.9)
    ax.set_xlabel("Z-value")
    ax.set_title(
        f"{title}\n({sig_mask.sum()} sig. regions, {100*sig_mask.mean():.1f}%)",
        fontweight='bold', fontsize=10)
    ax.set_ylabel("Region rank (by |Z|)" if ax == axes[0] else "")
    ax.grid(axis='x', alpha=0.3)
    ax.set_yticks([])
    ax.legend(handles=[Patch(facecolor='#D32F2F', label='EP increase'),
                        Patch(facecolor='#1565C0', label='EP decrease')],
              fontsize=8, loc='lower right')

plt.tight_layout(pad=1.5)
plt.savefig(f"{OUT_DIR}/Fig4_brain_region_EP_changes.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved Fig4")


# ════════════════════════════════════════════════════════════
# CELL 11 — Figure 5: EP Distribution Shift & Heterogeneity
# ════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.patch.set_facecolor('#FAFAFA')

# ─── Left: overlaid EP histograms ────────────────────────
ax = axes[0]
bins = np.linspace(0, ep_matrix.max() * 1.05, 80)
for i in range(4):
    ep_flat = ep_matrix[df_valid['group'].values == i].flatten()
    ax.hist(ep_flat, bins=bins, density=True, alpha=0.45,
            color=GROUP_COLORS[i],
            label=GROUP_LABELS_SHORT[i]+' yrs', edgecolor='none')
ax.set_xlabel("Energy Probability (EP)")
ax.set_ylabel("Density")
ax.set_title("EP Distribution: Concentration → Dispersion with Age\n"
             "(older groups show wider, flatter distributions)", fontweight='bold')
ax.legend(title="Age group", fontsize=9)
ax.grid(alpha=0.25)

# ─── Right: EP heterogeneity (std across regions) vs group ─
ax2 = axes[1]
ep_stds = [ep_matrix[df_valid['group'].values == i].std(axis=1).mean()
           for i in range(4)]
bars = ax2.bar(range(4), ep_stds, color=GROUP_COLORS, alpha=0.82,
               edgecolor='white', linewidth=1.5, width=0.6)
for i, v in enumerate(ep_stds):
    ax2.text(i, v * 1.015, f'{v:.6f}', ha='center', va='bottom', fontsize=9)

ax2.set_xticks(range(4))
ax2.set_xticklabels(GROUP_LABELS_SHORT)
ax2.set_xlabel("Age Group (years)")
ax2.set_ylabel("Mean EP std across regions")
ax2.set_title("Functional Connectome Heterogeneity vs Age\n"
              "(↑ heterogeneity = more specialised = ↓ NEE)", fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout(pad=2)
plt.savefig(f"{OUT_DIR}/Fig5_EP_distribution_shift.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved Fig5")


# ════════════════════════════════════════════════════════════
# CELL 12 — Figure 6: Regression Diagnostics
# ════════════════════════════════════════════════════════════

predicted = model.fittedvalues.values
residuals = model.resid.values
actual    = y.values
age_reg   = df_reg['age'].values

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor('#FAFAFA')
fig.suptitle("Regression: NEE ~ age + age² + sex + meanFD + site",
             fontsize=13, fontweight='bold')

# ─ Actual vs Predicted ─
ax = axes[0]
ax.scatter(actual, predicted, alpha=0.3, s=12, color='steelblue')
lims = [min(actual.min(), predicted.min()), max(actual.max(), predicted.max())]
ax.plot(lims, lims, 'r--', lw=1.5)
ax.set_xlabel("Actual NEE")
ax.set_ylabel("Predicted NEE")
ax.set_title(f"Actual vs Predicted  (R²={r2:.3f})", fontweight='bold')
ax.grid(alpha=0.25)

# ─ Residuals vs Age ─
ax2 = axes[1]
ax2.scatter(age_reg, residuals, alpha=0.3, s=12, color='darkorange')
ax2.axhline(0, color='black', lw=1.2)
ax2.set_xlabel("Age (years)")
ax2.set_ylabel("Residual")
ax2.set_title("Residuals vs Age", fontweight='bold')
ax2.grid(alpha=0.25)

# ─ Coefficient plot ─
ax3 = axes[2]
params_of_interest = ['age', 'age2', 'sex', 'meanFD']
param_labels       = ['Age (β₁)', 'Age² (β₂)', 'Sex (β₃)', 'MeanFD (β₄)']
betas    = [model.params[p]         for p in params_of_interest]
ci_low   = [model.conf_int().loc[p, 0] for p in params_of_interest]
ci_high  = [model.conf_int().loc[p, 1] for p in params_of_interest]
p_reg    = [model.pvalues[p]        for p in params_of_interest]
bar_cols = ['#D32F2F' if p<0.05 else '#90A4AE' for p in p_reg]

ax3.barh(range(4), betas, color=bar_cols, alpha=0.82, edgecolor='white')
ax3.errorbar(betas, range(4),
             xerr=[np.array(betas)-np.array(ci_low),
                   np.array(ci_high)-np.array(betas)],
             fmt='none', color='black', capsize=4, lw=1.5)
ax3.axvline(0, color='black', lw=1)
ax3.set_yticks(range(4))
ax3.set_yticklabels(param_labels, fontsize=10)
ax3.set_xlabel("β coefficient (95% CI)")
ax3.set_title("Regression Coefficients\n(red = p<0.05)", fontweight='bold')
for i, (b, p) in enumerate(zip(betas, p_reg)):
    star = '*' if p<0.05 else ''
    ax3.text(b + abs(b)*0.05, i, f'p={p:.3f}{star}', va='center', fontsize=8.5)
ax3.grid(axis='x', alpha=0.3)

plt.tight_layout(pad=1.5)
plt.savefig(f"{OUT_DIR}/Fig6_regression_analysis.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved Fig6")


# ════════════════════════════════════════════════════════════
# CELL 14 — Final Summary Print
# ════════════════════════════════════════════════════════════

print("=" * 65)
print("  ANALYSIS COMPLETE — KEY RESULTS")
print("=" * 65)
print(f"  Subjects processed : {len(df_valid)}")
print(f"  Age range          : {df_valid['age'].min():.1f} – {df_valid['age'].max():.1f} years")
print(f"  Brain parcels      : {N_REGIONS} (Glasser HCP)")
print()
print(f"  NEE trend with age : β={beta_age:.4f}, p={p_age:.4e}")
print(f"  Quadratic term     : β²={beta_age2:.4f}, p={p_age2:.4e}")
print(f"  Polynomial R²      : {r2_poly:.3f}")
print(f"  MLR model R²       : {r2:.3f}")
print()
print("  Group NEE means:")
for i in range(4):
    lbl = GROUP_DEFS[i][0].replace('\n',' ')
    print(f"    {lbl}: {group_dfs[i]['NEE'].mean():.4f} ± {group_dfs[i]['NEE'].std():.4f}")
print()
print("  Mann-Whitney pairwise:")
for (i,j) in pair_keys:
    p = mw_results[(i,j)]
    print(f"    G{i+1} vs G{j+1}: p = {p:.4e}  "
          f"{'***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'}")
print()
print("  Significant EP regions (FDR q<0.05):")
for lbl, sig in [("G1 vs G2", sig12),("G2 vs G3", sig23),("G3 vs G4", sig34)]:
    print(f"    {lbl}: {sig.sum()} / {N_REGIONS}  ({100*sig.mean():.1f}%)")
print()
print(f"  Outputs saved to: {OUT_DIR}")
print("=" * 65)

MessageError: Error: credential propagation was unsuccessful